# Optimisation des vacations du bloc opératoire par recuit simulé

**Projet fil rouge — rapport complet de l'étude**

## Objectif
Proposer une **nouvelle grille de vacations** (qui opère dans quelle salle, quel jour, à quelle heure) qui **lisse l'occupation des lits** et de l'unité ambulatoire, en respectant :
- une grille **applicable toute l'année** (cycle de 4 semaines : paire, impaire A, paire, impaire B, comme la grille actuelle) ;
- des chirurgiens qui **gardent environ leur temps de travail** et qui **continuent tous à travailler à la clinique** (tous ceux de la grille actuelle gardent des créneaux) ;
- **moins de lits le week-end, et encore moins pendant les vacances scolaires**.

## Démarche en deux niveaux
1. **Niveau patients** : pour une grille donnée, un recuit simulé choisit la date de chaque patient (à ±30 jours de sa date réelle) pour lisser les lits. On y choisit la **fonction coût**.
2. **Niveau grille** : un second recuit simulé réattribue les créneaux de la grille aux chirurgiens. Chaque grille candidate est évaluée par un modèle « attendu » (rapide), puis les meilleures grilles sont **validées en rejouant des années réelles** (2022, et 2021 hors échantillon).

## Plan
0. Réglages et vérifications
1. Les données
2. La fonction coût du niveau patients (comparaison de 4 fonctions)
3. Le recuit au niveau patients : poids, paramètres, convergence, taille
4. Le recuit au niveau grille : grilles proposées A et B
5. Réglage du recuit sur la grille (T0, α, budget, comparaison à une descente)
6. Stabilité : combien de créneaux changer ? Le modèle attendu prédit-il bien ?
7. Validation sur 2022 et 2021, avec répétitions et tests statistiques
8. Sensibilité aux hypothèses
9. Temps de calcul
10. Conclusions et limites

**Durée** : `RAPIDE = True` ≈ 30 à 45 min ; `RAPIDE = False` (étude complète) ≈ 1 h 30 à 2 h. Tous les résultats sont aussi enregistrés dans `resultats_notebook/`, `resultats_recuit/`.

In [3]:
import os
print(os.getcwd())
print(os.listdir())

import shutil, glob, os
for f in glob.glob("notebook_recuit/*.py"):
    shutil.copy(f, ".")
print(sorted(f for f in os.listdir() if f.endswith(".py")))

c:\Users\ChristopheGambier\Downloads\recuit
['exemple_execution_rapide.html', 'notebook_recuit', 'rapport_recuit_grille.ipynb']
[]


## 0. Réglages et vérifications

Fichiers nécessaires dans le même dossier : `modele.py` (code principal du groupe), `donnees_reelles.py`, `fonctions_cout.py`, `recuit.py`, `etude_recuit.py`, `grille_vacations.py`, `recuit_grille.py`, `analyse.py` et la base `donees_bloc_anonyme_pour_centrale_2026.xlsx`.

In [1]:
RAPIDE = False          # True : version courte pour tester (~30-45 min)

import sys, time, json, math, warnings, argparse
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

fichiers = ["modele.py", "donnees_reelles.py", "fonctions_cout.py", "recuit.py", "etude_recuit.py",
            "grille_vacations.py", "recuit_grille.py", "analyse.py", "donees_bloc_anonyme_pour_centrale_2026.xlsx"]
manquants = [f for f in fichiers if not Path(f).exists()]
assert not manquants, f"Fichiers manquants : {manquants}"

# --- réglages de l'étude -------------------------------------------------------
R = dict(
    graines_validation=[0, 1] if RAPIDE else [0, 1, 2, 3, 4],
    annees=[2022, 2021],
    restarts=2 if RAPIDE else 5,
    iter_patients=60_000 if RAPIDE else 150_000,
    t0=[0.1, 1, 5],
    alphas=[0.99, 0.995] if RAPIDE else [0.99, 0.995, 0.998],
    graines_parametres=[0] if RAPIDE else [0, 1],
    poids_stabilite=[0, 0.6, 1.5, 3] if RAPIDE else [0, 0.3, 0.6, 1, 1.5, 2, 3],
    graines_stabilite=[0] if RAPIDE else [0, 1],
    budgets=[10_000, 50_000, 100_000] if RAPIDE else [10_000, 25_000, 50_000, 100_000, 200_000],
)
TEMPS = {}            # durée de chaque partie, pour la partie 9
class Chrono:
    def __init__(self, nom): self.nom = nom
    def __enter__(self): self.t = time.perf_counter()
    def __exit__(self, *a):
        TEMPS[self.nom] = time.perf_counter() - self.t
        print(f"[{self.nom} : {TEMPS[self.nom] / 60:.1f} min]")

import modele
assert hasattr(modele, "CoutTotal") and hasattr(modele, "Solution"), "modele.py du groupe attendu"
from donnees_reelles import charger_base, construire_rejeu
from fonctions_cout import Evaluateur, candidates, verifier_coherence
from grille_vacations import Grille, SEMAINES_ACTIVES
from recuit_grille import (ModeleAttendu, PoidsGrille, ParamsRecuitGrille, EvaluationGrille, recuit_grille,
                           heures_selon_activite, NUITS)
import analyse as A
A.dossiers()
print("Mode :", "RAPIDE" if RAPIDE else "COMPLET")
print(R)

AssertionError: Fichiers manquants : ['modele.py', 'donnees_reelles.py', 'fonctions_cout.py', 'recuit.py', 'etude_recuit.py', 'grille_vacations.py', 'recuit_grille.py', 'analyse.py', 'donees_bloc_anonyme_pour_centrale_2026.xlsx']

## 1. Les données

**La base** : une ligne par intervention (2019-2022) — chirurgien, type d'intervention, dates d'entrée, d'opération et de sortie, heures d'entrée et de sortie de salle.

Choix de préparation :
- **nuits = dates** (sortie − entrée), et non la colonne « durée de séjour », fausse pour ~1 900 séjours ; les entrées la veille sont comptées ;
- **ambulatoire ⇔ zéro nuit** ;
- durée opératoire prévue = médiane du type d'intervention ; marge de sécurité = P90 − médiane (comme dans `modele.py`) ; heures 00:00 = inconnues.

**La grille actuelle** : la grille « Version 28 » transmise par l'hôpital (salles 2 à 5, semaines paires / impaires, créneaux URGENCES et LIBRE).

In [ ]:
with Chrono("1. données"):
    base = charger_base()
    G = Grille.actuelle()
    mod = ModeleAttendu(base, G)
    hosp = base[base.nuits > 0]
    display(pd.Series({
        "interventions": len(base),
        "période": f"{base.inter.min():%d/%m/%Y} → {base.inter.max():%d/%m/%Y}",
        "interventions en 2022": int((base.inter.dt.year == 2022).sum()),
        "part d'ambulatoire": f"{(base.nuits == 0).mean():.1%}",
        "hospitalisés entrés la veille": f"{(hosp.nuits_pre > 0).mean():.1%}",
        "nuits par séjour hospitalisé (médiane)": hosp.nuits.median(),
        "durée opératoire médiane (min)": base.tros.median(),
    }).to_frame("valeur"))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 3.8))
hosp.nuits.clip(upper=15).value_counts().sort_index().plot.bar(ax=axs[0])
axs[0].set_title("Nuits par séjour hospitalisé (15 = 15 et +)")
jours = ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"]
base[base.inter.dt.year == 2022].inter.dt.dayofweek.value_counts().sort_index().rename(index=dict(enumerate(jours))).plot.bar(ax=axs[1])
axs[1].set_title("Interventions 2022 par jour")
act = base[base.inter.dt.year == 2022].chir.value_counts()
act.plot.bar(ax=axs[2])
axs[2].set_title("Interventions 2022 par chirurgien")
plt.tight_layout(); A.sauver_fig(fig, "1_donnees.png"); plt.show()

### 1.1 La réalité de 2022 : l'occupation des lits

Profil réel des lits en 2022 (patients à leur vraie date, dans les créneaux réellement utilisés). La cible (pointillés) répartit la charge moyenne selon le personnel : **85 % les nuits de week-end** (vendredi, samedi, dimanche) et **70 % pendant les vacances scolaires (zone B), ponts et fériés**.

In [ ]:
rr, ev_reel = A.simuler_reel(base, 2022)
display(pd.Series(ev_reel.indicateurs()).round(2).to_frame("réel 2022"))
fig = A.fig_profils({(2022, "réel"): ev_reel}, 2022); A.sauver_fig(fig, "1_profil_reel_2022.png"); plt.show()

### 1.2 La grille actuelle et le temps de chaque chirurgien

On compare les heures de bloc par semaine de la grille actuelle au **besoin** de chaque chirurgien, calculé sur son activité réelle 2022 (durées réelles + 15 min d'inter-salle, vacations remplies à 85 %).

In [ ]:
h0 = G.heures_par_semaine()
cap = pd.DataFrame({"heures/sem (grille actuelle)": h0, "besoin 2022 (h/sem)": mod.besoin_h,
                    "patients/sem 2022": mod.demande}).round(1)
cap["besoin / temps"] = (cap["besoin 2022 (h/sem)"] / cap["heures/sem (grille actuelle)"]).round(2)
display(cap.sort_values("besoin / temps", ascending=False))
display(Markdown("**Lecture** : un rapport > 1 signifie que le chirurgien n'a pas assez de temps de bloc pour son "
                 "activité 2022 ; < 1, qu'il en a plus que nécessaire. DEVOS, DS et DN n'ont pas d'activité dans la "
                 "base (arrivés depuis ?) : ils gardent leurs créneaux."))

## 2. Niveau patients : quelle fonction coût pour lisser les lits ?

On rejoue le **1er semestre 2022** (patients réels, créneaux réellement utilisés). Le recuit part du planning réel et peut déplacer chaque patient de ±30 jours dans un créneau de son chirurgien. Quatre fonctions coût sont comparées **sur les mêmes indicateurs physiques** :

| | Fonction coût | Principe |
|---|---|---|
| C0 | groupe (`CoutTotal` de `modele.py`) | variance des lits, des places, des remplissages + délai, poids 1 |
| C1 | groupe normalisée | mêmes termes divisés par leur valeur sur le planning réel, poids lits 40 %, places 20 %, bloc 20 %, délai 20 % |
| C2 | **cible lits** | écart à une cible nuit par nuit (week-end 85 %, vacances 70 %) + dépassement de ±2 lits ; ambulatoire, temps de bloc perdu, délai ; termes normalisés |
| C3 | cible lits seule | uniquement le terme lits de C2 |

Contraintes dures pour toutes : lits et ambulatoires en surnombre, dépassement des vacations (pénalité 10⁶).

In [ ]:
with Chrono("2-3. niveau patients"):
    import etude_recuit as ER
    args = argparse.Namespace(debut="2022-01-03", semaines=25, fenetre=30, graines=2 if RAPIDE else 3,
                              iterations=50_000 if RAPIDE else 80_000, rapide=RAPIDE,
                              poids_lits=[0.1, 0.4, 0.8, 1.0] if RAPIDE else [0.1, 0.25, 0.4, 0.6, 0.8, 1.0],
                              t0=[0.1, 0.5, 5] if RAPIDE else [0.1, 0.5, 2, 5],
                              alphas=[0.99, 0.995] if RAPIDE else [0.99, 0.995, 0.998],
                              tailles_semaines=[8, 25] if RAPIDE else [8, 13, 25])
    rj1 = ER.rejeu(args)
    ev1 = ER.evaluateur(rj1)
    print(f"Cohérence avec CoutTotal de modele.py : écart {verifier_coherence(ev1):.1e} (doit être ~0)")
    for etape in (ER.etape_reel, ER.etape_fonctions, ER.etape_poids, ER.etape_parametres,
                  ER.etape_convergence, ER.etape_taille):
        etape(args)
    ER.synthese(args)

In [ ]:
fo = pd.read_csv("resultats_recuit/fonctions.csv", sep=";")
reel1 = pd.read_csv("resultats_recuit/reel.csv", sep=";")
ind1 = ["lits_ecart_type", "lits_pic", "nuits_hors_2lits_pct", "ratio_weekend_semaine", "ambu_ecart_type",
        "bloc_remplissage_pct", "decalage_moyen_j", "temps"]
t = A.tableau_ic(fo, "fonction", ind1)
display(Markdown("**Indicateurs physiques selon la fonction coût** (moyenne ± IC 95 %) — réel 1er semestre 2022 :"))
display(reel1[[c for c in ind1 if c in reel1]].round(2))
display(t)
for f in ["2_profils_par_fonction_cout.png", "3_comparaison_fonctions_cout.png"]:
    display(Image(filename=f"resultats_recuit/figures/{f}"))

**Lecture** : C0 (fonction actuelle du groupe) lisse moins bien et remplit davantage le week-end (variance sur une capacité constante, termes non normalisés : le délai écrase les lits). **C2** est la seule qui améliore tous les indicateurs à la fois ; C3 lisse le plus les lits mais dégrade l'ambulatoire. **C2 est retenue** pour la suite.

## 3. Niveau patients : poids, paramètres, convergence, taille

In [ ]:
for f in ["4_compromis_poids.png", "5_parametres_T0_alpha.png", "6_qualite_temps.png", "7_convergence.png",
          "8_taille_population.png"]:
    display(Image(filename=f"resultats_recuit/figures/{f}"))
pa = pd.read_csv("resultats_recuit/parametres.csv", sep=";")
display(Markdown("**Paramètres du recuit patients** (moyennes) :"))
display(pa.groupby(["T0_facteur", "alpha"])[["lits_ecart_type", "nuits_hors_2lits_pct", "temps", "iterations"]].mean().round(2))

**Lecture** : un poids de 25 à 40 % sur les lits donne le meilleur compromis ; le **refroidissement lent** (α proche de 1) améliore le lissage au prix du temps ; avec un budget d'itérations proportionnel au nombre de patients, la qualité reste stable quand la population grandit.

## 4. Niveau grille : proposer une nouvelle grille de vacations

**Décision** : qui occupe chacun des 33 créneaux modifiables de la grille (salles, horaires, cycle de 4 semaines, créneaux URGENCES et LIBRE inchangés).
**Mouvements** : échanger les occupants de deux créneaux (80 %), ou donner un créneau à un autre chirurgien (20 %).

**Évaluation rapide (« modèle attendu »)** : à partir de la base, pour chaque chirurgien, sa demande (patients/semaine en 2022, répartie sur ses créneaux au prorata de leur durée), la probabilité qu'un de ses patients occupe encore un lit s nuits après l'opération, et sa part d'ambulatoire. On obtient les lits et ambulatoires **attendus** chaque nuit du cycle de 28 jours.

**Fonction coût de la grille** (termes lits et ambulatoire normalisés par la grille actuelle) :
- 0,7 × écart des lits attendus à la cible (nuits de week-end à 85 %) ;
- 0,3 × variance des ambulatoires attendus par jour ouvré ;
- 50 × (écart de temps de travail au-delà de ±10 %)² par chirurgien ;
- 20 × (heures manquantes / heures nécessaires)² : on ne retire pas de temps à un chirurgien qui en manque ;
- 1,0 × part des créneaux modifiés (stabilité) ;
- 10⁶ × conflits (un chirurgien à deux endroits) + 10⁶ × chirurgien sans aucun créneau.

**Deux scénarios** :
- **Grille A** : chaque chirurgien garde son temps actuel (±10 %) — c'est la contrainte du projet ;
- **Grille B** : même temps total, redistribué selon l'activité 2022, chaque chirurgien gardant **au moins 50 %** de son temps actuel.

Le recuit est aléatoire : on le lance plusieurs fois et on garde la meilleure grille.

In [ ]:
with Chrono("4. grilles A et B"):
    cible_B = heures_selon_activite(base, G)
    print("Grille A (temps conservé ±10 %)")
    best_A, runs_A = A.recuits_grille(G, mod, R["restarts"])
    print("Grille B (temps rééquilibré, au moins 50 % du temps actuel)")
    best_B, runs_B = A.recuits_grille(G, mod, R["restarts"], heures_cible=cible_B)
GA_, GB_ = best_A.grille, best_B.grille
GA_.nom, GB_.nom = "A", "B"
grilles = {"actuelle": G, "A": GA_, "B": GB_}
for nom, g in (("grille_A", GA_), ("grille_B", GB_)):
    g.exporter_excel(A.SORTIE / f"{nom}.xlsx", reference=G)
    (A.SORTIE / f"{nom}.json").write_text(json.dumps(g.chirurgiens), encoding="utf-8")
runs = pd.DataFrame([{"grille": n, "exécution": i, "coût final": r.cout_final, "itérations": r.iterations,
                      "temps (s)": round(r.temps, 1),
                      "créneaux modifiés": sum(a != b for a, b in zip(r.grille.chirurgiens, G.chirurgiens))}
                     for n, rs in (("A", runs_A), ("B", runs_B)) for i, r in enumerate(rs)])
display(Markdown("**Exécutions du recuit** (la meilleure de chaque grille est retenue) :"))
display(runs.round(3))

### 4.1 Décomposition de la fonction coût

Ce que le recuit a « échangé » entre ses objectifs. Lits et ambulatoire en valeur relative (grille actuelle = 1) ; les termes de contrainte doivent rester nuls ou faibles.

In [ ]:
display(Markdown("**Avec l'objectif de temps de la grille A (temps actuel)** :"))
display(A.decomposition(grilles, mod, G).round(3))
display(Markdown("**Avec l'objectif de temps de la grille B (temps rééquilibré)** :"))
display(A.decomposition(grilles, mod, G, heures_cible=cible_B).round(3))

### 4.2 Temps de travail : tous les chirurgiens gardent des créneaux

In [ ]:
th = A.heures_table(grilles, mod)
display(th)
absents = {n: [m for m in h0 if g.heures_par_semaine().get(m, 0) == 0] for n, g in grilles.items()}
conflits = {n: len(g.conflits()) for n, g in grilles.items()}
display(Markdown(f"**Chirurgiens sans créneau** : {absents} — **conflits** : {conflits}"))
fig = A.fig_heures(th); A.sauver_fig(fig, "4_temps_par_chirurgien.png"); plt.show()

### 4.3 Les grilles proposées
En orange : créneaux dont l'occupant change. Les fichiers Excel `resultats_notebook/grille_A.xlsx` et `grille_B.xlsx` reprennent la présentation de la grille de l'hôpital (créneaux modifiés marqués d'une étoile).

In [ ]:
for nom, g in (("proposée A", GA_), ("proposée B", GB_)):
    fig = A.fig_grille(G, g, nom); A.sauver_fig(fig, f"4_grille_{nom[-1]}.png"); plt.show()

### 4.4 Lits et ambulatoires attendus sur le cycle

In [ ]:
fig = A.fig_lits_attendus(grilles, mod, G); A.sauver_fig(fig, "4_lits_attendus.png"); plt.show()

## 5. Réglage du recuit sur la grille

### 5.1 Température initiale T0 et refroidissement α
T0 est exprimée en multiple de la dégradation **typique** d'un mouvement (1er décile des dégradations : la plupart des mouvements cassent le temps de travail d'un chirurgien, et calibrer sur la moyenne donnerait une température trop haute).

In [ ]:
with Chrono("5. paramètres de la grille"):
    lignes, conv = [], []
    for t0 in R["t0"]:
        for a in R["alphas"]:
            for g in R["graines_parametres"]:
                r = recuit_grille(G, mod, ParamsRecuitGrille(T0_facteur=t0, alpha=a, graine=g, max_iter=250_000))
                lignes.append({"T0": t0, "alpha": a, "graine": g, "coût": r.cout_final, "itérations": r.iterations,
                               "temps (s)": r.temps})
                if g == R["graines_parametres"][0] and a == 0.995:
                    conv += [{"config": f"T0 = {t0}", **j} for j in r.journal]
            print(f"   T0 = {t0}, alpha = {a}")
    par_g = pd.DataFrame(lignes); par_g.to_csv(A.SORTIE / "parametres_grille.csv", sep=";", index=False)
    conv = pd.DataFrame(conv)

fig, axs = plt.subplots(1, 3, figsize=(17, 4.3))
for ax, (col, titre, cmap) in zip(axs[:2], (("coût", "Coût final (plus bas = mieux)", "RdYlGn_r"),
                                            ("temps (s)", "Temps de calcul (s)", "Blues"))):
    piv = par_g.groupby(["alpha", "T0"])[col].mean().unstack()
    im = ax.imshow(piv.values, cmap=cmap, aspect="auto", origin="lower")
    ax.set_xticks(range(len(piv.columns)), piv.columns); ax.set_yticks(range(len(piv.index)), piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, f"{piv.values[i, j]:.3f}" if col == "coût" else f"{piv.values[i, j]:.0f}", ha="center", va="center")
    ax.set_xlabel("T0 (x dégradation typique)"); ax.set_ylabel("alpha"); ax.set_title(titre); fig.colorbar(im, ax=ax)
for nom, g in conv.groupby("config", sort=False):
    axs[2].plot(g.iteration, g.meilleur, label=nom)
axs[2].set_xlabel("itération"); axs[2].set_ylabel("meilleur coût"); axs[2].set_title("Convergence (α = 0,995)"); axs[2].legend()
plt.tight_layout(); A.sauver_fig(fig, "5_parametres_grille.png"); plt.show()
display(par_g.groupby(["T0", "alpha"])[["coût", "itérations", "temps (s)"]].mean().round(3))

### 5.2 Qualité en fonction du budget, et comparaison à une descente locale
Pour chaque budget d'itérations, α est calé pour que la température atteigne Tmin exactement à la fin. La **descente locale** (température quasi nulle : on n'accepte que les améliorations) montre l'intérêt d'accepter des dégradations.

In [ ]:
with Chrono("5. budget et descente"):
    lignes = []
    for budget in R["budgets"]:
        for methode, t0 in (("recuit simulé", 1.0), ("descente locale", 1e-4)):
            for g in R["graines_parametres"]:
                tmin = 0.005 if t0 >= 1 else 1e-6
                alpha = (tmin / t0) ** (100 / budget)
                r = recuit_grille(G, mod, ParamsRecuitGrille(T0_facteur=t0, Tmin_facteur=tmin, alpha=alpha,
                                                             max_iter=budget, graine=g))
                lignes.append({"budget": budget, "méthode": methode, "graine": g, "coût": r.cout_final, "temps (s)": r.temps})
        print(f"   budget {budget}")
    bud = pd.DataFrame(lignes)
fig, ax = plt.subplots(figsize=(8, 4.3))
for m, g in bud.groupby("méthode"):
    a = g.groupby("budget").agg(c=("coût", "mean"), t=("temps (s)", "mean"))
    ax.plot(a.t, a.c, marker="o", label=m)
    for b, row in a.iterrows():
        ax.annotate(f"{b // 1000}k", (row.t, row.c), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("temps de calcul (s)"); ax.set_ylabel("coût final (plus bas = mieux)")
ax.set_title("Recuit simulé contre descente locale, selon le budget d'itérations"); ax.legend()
A.sauver_fig(fig, "5_budget_descente.png"); plt.show()
display(bud.groupby(["méthode", "budget"])[["coût", "temps (s)"]].mean().round(3))

### 5.3 Variabilité d'une exécution à l'autre

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.boxplot([[r.cout_final for r in runs_A], [r.cout_final for r in runs_B]])
ax.set_xticks([1, 2], ["grille A", "grille B"]); ax.set_ylabel("coût final"); ax.set_title(f"{R['restarts']} exécutions du recuit")
A.sauver_fig(fig, "5_variabilite.png"); plt.show()

## 6. Stabilité : combien de créneaux faut-il changer ? Le modèle attendu prédit-il bien ?

On fait varier le poids de la stabilité. Pour chaque grille obtenue, on compare le **gain prévu** par le modèle attendu au **gain mesuré** en rejouant 2022 (une simulation par grille).

In [ ]:
with Chrono("6. stabilité"):
    ev0 = EvaluationGrille(G.copie(), mod, PoidsGrille(), G); t0g = ev0.termes_bruts()
    _, ev_act, _ = A.simuler_grille(G, base, 2022, 0, iter_patients=R["iter_patients"])
    ref_mes = ev_act.indicateurs()
    lignes = []
    for w in R["poids_stabilite"]:
        for g in R["graines_stabilite"]:
            r = recuit_grille(G, mod, ParamsRecuitGrille(graine=g), poids=PoidsGrille(changements=w))
            t = EvaluationGrille(r.grille.copie(), mod, PoidsGrille(), G).termes_bruts()
            _, ev, _ = A.simuler_grille(r.grille, base, 2022, 0, iter_patients=R["iter_patients"])
            m = ev.indicateurs()
            lignes.append({"poids stabilité": w, "graine": g,
                           "créneaux modifiés": sum(a != b for a, b in zip(r.grille.chirurgiens, G.chirurgiens)),
                           "gain prévu lits (%)": 100 * (1 - t["lits"] / t0g["lits"]),
                           "gain prévu ambu (%)": 100 * (1 - t["ambu"] / t0g["ambu"]),
                           "gain mesuré nuits hors ±2 (points)": ref_mes["nuits_hors_2lits_pct"] - m["nuits_hors_2lits_pct"],
                           "gain mesuré écart-type ambu (%)": 100 * (1 - m["ambu_ecart_type"] / ref_mes["ambu_ecart_type"])})
        print(f"   poids {w}")
    stab = pd.DataFrame(lignes); stab.to_csv(A.SORTIE / "stabilite.csv", sep=";", index=False)
display(stab.round(2))
fig, axs = plt.subplots(1, 3, figsize=(17, 4.3))
sc = axs[0].scatter(stab["créneaux modifiés"], stab["gain prévu lits (%)"], c=stab["poids stabilité"], cmap="viridis", label="lits")
axs[0].scatter(stab["créneaux modifiés"], stab["gain prévu ambu (%)"], c=stab["poids stabilité"], cmap="viridis", marker="^", label="ambulatoire")
axs[0].set_xlabel("créneaux modifiés (sur 33)"); axs[0].set_ylabel("gain prévu (%)"); axs[0].set_title("Gain prévu selon le nombre de créneaux modifiés"); axs[0].legend()
fig.colorbar(sc, ax=axs[0], label="poids de la stabilité")
for ax, (x, y, titre) in zip(axs[1:], (("gain prévu lits (%)", "gain mesuré nuits hors ±2 (points)", "Lits : prévu contre mesuré"),
                                       ("gain prévu ambu (%)", "gain mesuré écart-type ambu (%)", "Ambulatoire : prévu contre mesuré"))):
    ax.scatter(stab[x], stab[y]); ax.set_xlabel(x); ax.set_ylabel(y)
    rho = stab[[x, y]].corr(method="spearman").iloc[0, 1]
    ax.set_title(f"{titre} (corrélation de Spearman {rho:.2f})")
plt.tight_layout(); A.sauver_fig(fig, "6_stabilite_et_prediction.png"); plt.show()

**Lecture** : le graphique de gauche montre le compromis entre bouleversement de la grille et gain. Les deux graphiques de droite vérifient que le **modèle attendu est un bon guide** : si la corrélation est positive et nette, les grilles que le recuit juge meilleures le sont aussi quand on rejoue l'année réelle.

## 7. Validation sur des années réelles

On rejoue **toute l'année** avec les vrais patients, dans la grille actuelle et dans les grilles A et B, avec le recuit patients (fonction C2). La grille a été construite à partir de l'activité **2022** : **2021** est une validation **hors échantillon**. Chaque simulation est répétée avec plusieurs graines.

Deux séries d'indicateurs :
- **bruts** : tous les patients placés ;
- **à patients identiques** (`cmp_`) : on retire les patients restés sans créneau dans au moins une grille, pour comparer exactement les mêmes patients.

La colonne « réel » est ce que l'hôpital a vraiment fait (vraies dates, créneaux réellement utilisés), comme repère.

In [ ]:
with Chrono("7. validation"):
    profils = {}
    val = A.validation(grilles, base, R["annees"], R["graines_validation"], iter_patients=R["iter_patients"], profils=profils)
    val.to_csv(A.SORTIE / "validation.csv", sep=";", index=False)

In [ ]:
IND = {"lits_ecart_type": "Écart-type des lits", "lits_pic": "Pic de lits",
       "nuits_hors_2lits_pct": "Nuits hors ±2 lits (%)", "ambu_ecart_type": "Écart-type ambulatoire / jour",
       "bloc_remplissage_pct": "Remplissage du bloc (%)", "patients_sans_creneau": "Patients sans créneau",
       "lits_semaine": "Lits moyens semaine", "lits_weekend": "Lits moyens week-end", "lits_vacances": "Lits moyens vacances"}
for annee in R["annees"]:
    v = val[val.annee == annee]
    display(Markdown(f"### Année {annee} {'(construction de la grille)' if annee == 2022 else '(hors échantillon)'}"))
    display(Markdown("**Indicateurs bruts** (moyenne ± IC 95 %) :"))
    display(A.tableau_ic(v, "scenario", list(IND), IND))
    vc = v[v.scenario != "réel"]
    display(Markdown(f"**À patients identiques** ({int(vc.patients_exclus_cmp.max())} patients exclus car sans créneau dans au moins une grille) :"))
    cmp = {f"cmp_{k}": l for k, l in IND.items() if k != "patients_sans_creneau"}
    display(A.tableau_ic(vc, "scenario", list(cmp), cmp))
    lignes = []
    for col, lib in (("cmp_nuits_hors_2lits_pct", "nuits hors ±2 lits"), ("cmp_lits_ecart_type", "écart-type des lits"),
                     ("cmp_ambu_ecart_type", "écart-type ambulatoire")):
        for b in ("A", "B"):
            p, n = A.wilcoxon_apparie(vc, col, "actuelle", b)
            d = vc[vc.scenario == b][col].mean() - vc[vc.scenario == "actuelle"][col].mean()
            lignes.append({"indicateur": lib, "comparaison": f"actuelle → {b}", "différence moyenne": round(d, 2),
                           "p (Wilcoxon apparié)": round(p, 4) if p == p else "trop peu de répétitions",
                           "significatif (p < 0,05)": (p < 0.05) if p == p else "—"})
    display(Markdown("**Tests statistiques** (mêmes graines, à patients identiques) :"))
    display(pd.DataFrame(lignes))
    fig = A.fig_barres_ic(v, {k: l for k, l in IND.items() if k in list(IND)[:6]}, f"Validation {annee}")
    A.sauver_fig(fig, f"7_validation_{annee}.png"); plt.show()

In [ ]:
for annee in R["annees"]:
    v = val[val.annee == annee]
    agg = v.groupby("scenario", sort=False)[["lits_semaine", "lits_weekend", "lits_vacances"]].mean()
    ax = agg.plot.bar(figsize=(9, 3.8), rot=0)
    ax.set_ylabel("lits moyens"); ax.set_title(f"{annee} : lits en semaine, le week-end, en vacances")
    A.sauver_fig(ax.figure, f"7_semaine_weekend_vacances_{annee}.png"); plt.show()
    fig = A.fig_profils(profils, annee); A.sauver_fig(fig, f"7_profils_{annee}.png"); plt.show()

## 8. Sensibilité aux hypothèses

Les résultats dépendent d'hypothèses qu'on ne peut pas vérifier dans la base. On regarde si **le gain de la grille A sur la grille actuelle** résiste quand on les change (2022, une simulation par cas, à patients identiques).

In [ ]:
with Chrono("8. sensibilité"):
    cas = [("référence (±30 j, WE 85 %, vac. 70 %, marge 50 %)", {}),
           ("fenêtre ±15 jours", {"fenetre": 15}),
           ("fenêtre ±45 jours", {"fenetre": 45}),
           ("cibles WE 75 % / vacances 85 %", {"facteur_weekend": 0.75, "facteur_reduit": 0.85}),
           ("pas de cible réduite (100 % / 100 %)", {"facteur_weekend": 1.0, "facteur_reduit": 1.0}),
           ("marge de sécurité complète", {"facteur_marge": 1.0})]
    lignes = []
    for nom, kw in cas:
        fen = kw.pop("fenetre", 30)
        res = {}
        for gn, g in (("actuelle", G), ("A", GA_)):
            rj, ev, _ = A.simuler_grille(g, base, 2022, 0, fenetre=fen, iter_patients=R["iter_patients"], **kw)
            res[gn] = (rj, ev)
        union = set().union(*({p for p in rj.mobiles if ev.sol.affectation[p] is None} for rj, ev in res.values()))
        ind = {}
        for gn, (rj, ev) in res.items():
            for p in union:
                if ev.sol.affectation[p] is not None:
                    ev.deplacer(p, None)
            ev.recalibrer_cible(); ind[gn] = ev.indicateurs()
        lignes.append({"hypothèse": nom,
                       "nuits hors ±2 actuelle": ind["actuelle"]["nuits_hors_2lits_pct"], "nuits hors ±2 A": ind["A"]["nuits_hors_2lits_pct"],
                       "écart-type ambu actuelle": ind["actuelle"]["ambu_ecart_type"], "écart-type ambu A": ind["A"]["ambu_ecart_type"],
                       "patients sans créneau (union)": len(union)})
        print("   ", nom)
    sens = pd.DataFrame(lignes); sens.to_csv(A.SORTIE / "sensibilite.csv", sep=";", index=False)
sens["gain nuits (points)"] = sens["nuits hors ±2 actuelle"] - sens["nuits hors ±2 A"]
sens["gain ambu (%)"] = 100 * (1 - sens["écart-type ambu A"] / sens["écart-type ambu actuelle"])
display(sens.round(2))

## 9. Temps de calcul (mesurés sur cet ordinateur)

In [ ]:
tt = pd.Series(TEMPS).div(60).round(1).to_frame("minutes")
display(tt)
r = runs[runs.grille == "A"]
nb_ = lambda x: f"{x:,.0f}".replace(",", " ")
display(Markdown(f"- Recuit sur la grille : **{nb_((r['itérations'] / r['temps (s)']).mean())} itérations/s**, "
                 f"{r['temps (s)'].mean():.0f} s par exécution de {nb_(r['itérations'].mean())} itérations."))
v = val[val.scenario != "réel"]
display(Markdown(f"- Simulation d'une année (≈ 4 000 patients) avec le recuit patients ({nb_(R['iter_patients'])} itérations) : "
                 f"**{v.temps_s.mean():.0f} s** en moyenne."))
display(Markdown(f"- Temps total du notebook : **{sum(TEMPS.values()) / 60:.0f} min**."))

## 10. Conclusions

In [ ]:
v22 = val[(val.annee == 2022) & (val.scenario != "réel")]
v21 = val[(val.annee == 2021) & (val.scenario != "réel")]
m = lambda v, s, c: v[v.scenario == s][c].mean()
dec = A.decomposition(grilles, mod, G)
txt = f'''
**Grille proposée A** (temps conservé) : {sum(a != b for a, b in zip(GA_.chirurgiens, G.chirurgiens))} créneaux modifiés sur 33,
tous les chirurgiens gardent des créneaux, aucun conflit. Lits attendus {100 * (1 - dec.loc["A", "lits (écart à la cible, rel.)"]):.0f} % plus réguliers,
ambulatoire {100 * (1 - dec.loc["A", "ambulatoire (variance, rel.)"]):.0f} % plus régulier (modèle attendu).

**Validation 2022, à patients identiques** : nuits hors ±2 lits {m(v22, "actuelle", "cmp_nuits_hors_2lits_pct"):.1f} % (grille actuelle)
→ {m(v22, "A", "cmp_nuits_hors_2lits_pct"):.1f} % (A) → {m(v22, "B", "cmp_nuits_hors_2lits_pct"):.1f} % (B) ;
écart-type ambulatoire {m(v22, "actuelle", "cmp_ambu_ecart_type"):.2f} → {m(v22, "A", "cmp_ambu_ecart_type"):.2f} → {m(v22, "B", "cmp_ambu_ecart_type"):.2f}.

**Hors échantillon (2021)** : nuits hors ±2 lits {m(v21, "actuelle", "cmp_nuits_hors_2lits_pct"):.1f} % → {m(v21, "A", "cmp_nuits_hors_2lits_pct"):.1f} % (A) ;
écart-type ambulatoire {m(v21, "actuelle", "cmp_ambu_ecart_type"):.2f} → {m(v21, "A", "cmp_ambu_ecart_type"):.2f}.

**Capacité** : patients sans créneau en 2022 : {m(v22, "actuelle", "patients_sans_creneau"):.0f} (actuelle), {m(v22, "A", "patients_sans_creneau"):.0f} (A),
{m(v22, "B", "patients_sans_creneau"):.0f} (B). Le temps de bloc de certains chirurgiens ne suffit pas à leur activité :
la grille B, qui redistribue le temps tout en laissant à chacun au moins la moitié de son temps actuel, règle l'essentiel du problème.
'''
display(Markdown(txt))

### Ce qu'il faut retenir
1. **La fonction coût compte autant que l'algorithme** : la fonction initiale du groupe (C0) optimisait surtout le délai ; une fonction avec une **cible explicite nuit par nuit** et des **termes normalisés** (C2) améliore tous les indicateurs à la fois.
2. **Changer qui occupe les créneaux suffit** à rendre les lits et surtout l'ambulatoire plus réguliers, sans changer les salles ni les horaires, et en gardant le temps de chaque chirurgien.
3. **Quelques créneaux suffisent** : au-delà d'une quinzaine de créneaux modifiés, le gain progresse peu (partie 6).
4. **Le recuit bat la descente locale** à budget égal (partie 5.2) : accepter temporairement des dégradations est utile sur ce problème plein d'impasses (contraintes de temps de travail).
5. **La vraie limite est la répartition du temps** entre chirurgiens : avec leur temps actuel, certains ne peuvent pas absorber leur activité.

### Limites
- Le lien **salle ↔ spécialité / équipement** et les **jours de disponibilité** des chirurgiens ne sont pas modélisés : la grille proposée est une base de discussion à valider avec eux.
- La base ne contient pas les dates de consultation : on suppose que chaque patient peut être opéré à **±30 jours** de sa vraie date (partie 8 pour l'effet de ce choix).
- Les cibles (**85 %** le week-end, **70 %** en vacances) et la **marge de sécurité** (moitié de P90 − médiane) sont des hypothèses à valider avec l'hôpital.
- 2021 est une année marquée par la crise sanitaire : la validation hors échantillon est utile mais imparfaite.
- `modele.py` compte les nuits à partir du jour opératoire : une nuit de veille est décalée d'un jour (le nombre total de lits-nuits est juste).